In [0]:
%load_ext autoreload
%autoreload 2

In [0]:
from utils.silver_functions_gl import *
from pyspark.sql import functions as F

In [0]:
#Lecture Bronze
df = spark.table("workspace.final_project.bronze_orders") # A replacer par silver clean par suppliers quand existant

df_quarantine_orders = (
    df.limit(0)
    .withColumn("quarantine_reason", F.lit(None).cast("string"))
    .withColumn("quarantine_timestamp", F.lit(None).cast("timestamp"))
)

# Lecture Silver
df_suppliers = spark.table("workspace.final_project.bronze_suppliers") # A remplacer par silver quand existant


In [0]:
### ORDER_ID & SUPPLIER_ID EN INTEGER   

df_int, df_quarantine = clean_integer_column(df, "order_id", to_delete=True, accept_float=False)

df_int = clean_integer_column(df_int, "supplier_id", to_delete=False, accept_float=False)

df_quarantine_orders = df_quarantine_orders.unionByName(df_quarantine)



In [0]:
# NETTOYAGE DATE
df_int_clean = (
    df_int
    .withColumn("order_date", parse_date_col("order_date"))
    .withColumn("delivery_date_expected", parse_date_col("delivery_date_expected"))
    .withColumn("delivery_date_actual", parse_date_col("delivery_date_actual"))
)

In [0]:
### SUPPRIME DUPLICATS
df_without_duplicates, df_quarantine = quarantine_full_duplicates(df_int_clean, quarantine_reason="full_duplicated_line")

df_quarantine_orders = df_quarantine_orders.unionByName(df_quarantine)


In [0]:
### VERIFICATION COHERENCE DATES

# Verification delivery_date_expected > order_date
df_time_coherence, df_quarantine_time= validate_date_order(
    df_without_duplicates,
    date_late_col="delivery_date_expected",
    date_early_col="order_date",
    primary_key="order_id")

# Verification delivery_date_actual > order_date
df_time_coherence, df_quarantine_time_2 = validate_date_order(
    df_without_duplicates,
    date_late_col="delivery_date_actual",
    date_early_col="order_date",
    primary_key="order_id"
)

df_quarantine_orders = df_quarantine_orders.unionByName(df_quarantine_time)



In [0]:
### VERIFICATION COHERENCE SUPPLIER_ID

df_valid, df_orders_quarantine = validate_foreign_key(
    df_main=df_time_coherence,
    df_reference=df_suppliers,
    fk_col="supplier_id",
    pk_col="supplier_id",
    primary_key_main="order_id"
    )
df_quarantine_orders = df_quarantine_orders.unionByName(df_orders_quarantine)


In [0]:

### SEPARATION DE TABLE

# Split of items and orders dataset
# Select columns for orders
df_orders = df_valid.select(
    "order_id",
    "supplier_id",
    "order_date",
    "delivery_date_expected",
    "delivery_date_actual"
)

# Select columns for items
df_items = df_valid.select(
    "order_id",
    F.posexplode("items").alias("item_index", "item")
)

# Flatten items
df_items = df_items.select(
    "order_id",
    "item_index",
    "item.product_category",
    "item.quantity",
    "item.unit_price"
)

# Ajout identifiant unique pour chaque item avec hash
df_items = df_items.withColumn(
    "item_id",
    F.sha2(F.concat_ws("||", F.col("order_id"), F.col("item_index")), 256)
)

df_quarantine_items = (
    df_items.limit(0)
    .withColumn("quarantine_reason", F.lit(None).cast("string"))
    .withColumn("quarantine_timestamp", F.lit(None).cast("timestamp"))
)

In [0]:
# Gérer les doublons
df_items_dedup, df_quarantine = quarantine_full_duplicates(df_items, quarantine_reason="full_duplicated_line")

df_quarantine_items = df_quarantine_items.unionByName(df_orders_quarantine)

df_missing_items = missing_values_report(df_items_dedup)

display(df_missing_items)

df_int = clean_integer_column(df_items_dedup, "quantity", to_delete=False, accept_float=True)
